# 02 — Cointegration evidence for long-only decisions

This notebook starts the decision-support layer of Cobasket. The goal is not to produce an automatic trading command. It converts a cointegration relation into **relative-value evidence** for stocks that can actually be bought and sold through a long-only account.

The distinction is important:

- **Statistical evidence** describes what the model sees.
- **A recommendation policy** maps that evidence onto an action threshold chosen by the user.
- The present score is **not a calibrated probability**. Probability calibration requires repeated out-of-sample predictions and observed outcomes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from cobasket.data import DataManager
from cobasket.evidence import (
    RecommendationPolicy,
    cointegration_evidence,
    evidence_table,
    recommend_assets,
    recommendation_table,
)


## 1. Download a small candidate basket

Replace these symbols with a basket you want to examine. Two years is enough for a demonstration, although a longer history is useful when assessing stability.

In [ ]:
tickers = ["AAPL", "MSFT"]
manager = DataManager(cache_dir="../price_cache")
prices = manager.prices(tickers, period="2y")
prices.tail()


## 2. Convert the basket relation into evidence

For prices $P_i$ and fitted weights $w_i$, the spread is

$$S_t = \sum_i w_i P_{i,t}.$$

The spread z-score measures how far the current weighted combination lies from its recent mean. Mean reversion predicts a displacement in the opposite direction.

For each asset, Cobasket therefore constructs signed relative-value evidence proportional to

$$-z_t w_i.$$

This has a simple vector interpretation:

- the weight vector defines a direction in multi-stock price space;
- the z-score gives the displacement along that direction;
- reversing the displacement gives the direction expected under mean reversion;
- projecting that direction onto each stock tells us which stock appears relatively cheap or expensive within this basket.

This is **relative**, not absolute. A stock can look cheap compared with its basket partner while both companies are expensive by conventional valuation measures.

In [ ]:
result = cointegration_evidence(
    prices,
    window=60,
    min_trace_ratio=1.0,
)

print(f"Latest spread z-score: {result.latest_z_score:+.2f}")
print(f"Johansen trace ratio: {result.trace_ratio:.2f}")
evidence_table(result)


## 3. Interpret the evidence score and confidence

The **score** lies between $-1$ and $+1$:

- positive: relatively attractive for buying or adding;
- near zero: no strong relative-value preference;
- negative: relatively unattractive, suggesting waiting or reducing rather than opening a short position.

The **confidence** also lies between 0 and 1, but it is currently a heuristic strength measure based on:

1. how unusual the latest spread displacement is; and
2. how strongly the Johansen test supports cointegration.

It is not yet the probability that the stock will rise. Treat it more like a detection-quality diagnostic than a posterior probability.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
result.spread.plot(ax=axes[0], title="Cointegration spread")
result.z_score.plot(ax=axes[1], title="Rolling spread z-score")
axes[1].axhline(0, linewidth=1)
axes[1].axhline(2, linestyle="--", linewidth=1)
axes[1].axhline(-2, linestyle="--", linewidth=1)
plt.tight_layout()


## 4. Convert evidence into long-only recommendations

The action depends on whether you already own the stock:

| Evidence | Not currently held | Currently held |
|---|---|---|
| Strongly positive | Strong buy | Strong add |
| Moderately positive | Buy | Add |
| Near neutral | Watch | Hold |
| Moderately negative | Wait | Hold without adding |
| Strongly negative | Avoid buying | Consider reducing |

This avoids the short-selling interpretation. A negative signal does not instruct a Trading 212 Invest or ISA account to sell borrowed shares. It means the stock looks relatively expensive within this basket.

The thresholds are a **decision policy**, analogous to selecting a detection threshold after considering false-positive and false-negative costs.

In [ ]:
# Enter the quantities you currently own. Use an empty dictionary for none.
holdings = {
    "AAPL": 1.0,
    # "MSFT": 2.0,
}

policy = RecommendationPolicy(
    weak_threshold=0.25,
    strong_threshold=0.60,
    min_confidence=0.20,
)

recommendations = recommend_assets(
    result.asset_evidence,
    holdings=holdings,
    policy=policy,
)
recommendation_table(recommendations)


## 5. Read the explanation, not just the label

A recommendation label compresses several quantities into one word. Always inspect the explanation and diagnostics. At this stage, Cobasket should be treated like an analysis instrument: it reports a measured displacement and applies a user-defined decision rule, but it does not establish that a trade will be profitable.

In [ ]:
for recommendation in recommendations:
    print(f"{recommendation.ticker}: {recommendation.action}")
    print(f"  score={recommendation.score:+.3f}, confidence={recommendation.confidence:.3f}")
    print(f"  {recommendation.explanation}\n")


## 6. What comes next

The next methodological step is walk-forward calibration:

1. fit the basket using only information available at a historical date;
2. issue a score and recommendation;
3. measure what happened over a specified future horizon;
4. repeat through time and over many baskets;
5. estimate how often each score range led to a favourable outcome.

Only after that exercise should a score such as `0.7` be translated into a statement such as “70% probability of outperformance.”